# LLM Zero-Shot Eval on Train Samples

Sends competition prompts to LLMs via Gemini API, extracts answers, scores by type.

**Models tested:**
- Gemini 3.1 Pro (thinking enabled)
- Gemma 4 31B (dense)
- Gemma 4 26B-A4B (MoE — same architecture class as Nemotron)

In [ ]:
# === Config ===
API_KEY = dict(l.strip().split('=', 1) for l in open('kaggle.env', encoding='utf-8-sig') if '=' in l and not l.startswith('#'))['GEMINI_API_KEY']
MODEL = "gemini-3.1-pro-preview"
SAMPLES_PER_TYPE = 2          # change to 10+ for better stats
MAX_OUTPUT_TOKENS = 16384
THINKING_BUDGET = 16384

# All models to test
MODELS_TO_TEST = {
    "gemini-3.1-pro": "gemini-3.1-pro-preview",
    "gemma-4-31b": "gemma-4-31b-it",
    "gemma-4-26b-moe": "gemma-4-26b-a4b-it",
}

# DNS bypass
RESOLVE_IP = "142.251.210.106"

SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

In [ ]:
import pandas as pd
import subprocess, json, re, math, os
from pathlib import Path

# Load data
WORK_DIR = Path(os.getcwd())
train = pd.read_csv(WORK_DIR / 'train.csv')
print(f"Loaded {len(train)} rows")

# Classify
def classify(p):
    p = p.lower()
    if 'bit manipulation' in p: return 'bit_manipulation'
    elif 'encrypt' in p: return 'encryption'
    elif 'gravitational' in p: return 'gravity'
    elif 'numeral system' in p: return 'numeral_system'
    elif 'transformation rules' in p: return 'equation_transform'
    elif 'unit conversion' in p: return 'unit_conversion'
    return 'unknown'

train['qtype'] = train['prompt'].apply(classify)
print(train['qtype'].value_counts())

In [ ]:
# Stratified sample
samples = (
    train.groupby('qtype', group_keys=False)
    .apply(lambda x: x.sample(min(SAMPLES_PER_TYPE, len(x)), random_state=42))
    .reset_index(drop=True)
)
print(f"Sampled {len(samples)} rows:")
print(samples['qtype'].value_counts().to_string())

In [ ]:
# === Gemini API call via curl (bypasses DNS issue) ===

def call_gemini(prompt):
    body = json.dumps({
        "contents": [{"parts": [{"text": prompt + SUFFIX}]}],
        "generationConfig": {
            "temperature": 1.0,
            "maxOutputTokens": MAX_OUTPUT_TOKENS,
            "thinkingConfig": {"thinkingBudget": THINKING_BUDGET}
        }
    })
    result = subprocess.run([
        "curl", "-s", "--max-time", "180",
        "--resolve", f"generativelanguage.googleapis.com:443:{RESOLVE_IP}",
        f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={API_KEY}",
        "-H", "Content-Type: application/json",
        "-d", body
    ], capture_output=True, text=True)
    try:
        return json.loads(result.stdout)
    except:
        return {"error": result.stdout[:500]}


def extract_response_text(resp):
    """Extract non-thinking text from Gemini response."""
    try:
        parts = resp['candidates'][0]['content']['parts']
        texts = [p['text'] for p in parts if 'thought' not in p and p.get('text')]
        return '\n'.join(texts)
    except:
        return None


def extract_thinking_text(resp):
    """Extract thinking/thought text from Gemini response."""
    try:
        parts = resp['candidates'][0]['content']['parts']
        texts = [p['text'] for p in parts if p.get('thought') and p.get('text')]
        return '\n'.join(texts)
    except:
        return None


def extract_answer(text):
    """Extract answer from \\boxed{} with LaTeX cleanup."""
    if not text:
        return 'NOT_FOUND'
    # \boxed{\text{...}}
    m = re.findall(r'\\boxed\{\\text\{([^}]*)\}', text)
    if m:
        return m[-1].strip()
    # \boxed{...}
    m = re.findall(r'\\boxed\{([^}]*)\}', text)
    if m:
        ans = m[-1].strip()
        # LaTeX cleanup
        ans = ans.replace('\\ ', ' ')
        ans = ' '.join(ans.split())
        if '\\text' in ans:
            ans = ans.split('\\text')[0].strip()
        return ans
    return 'NOT_FOUND'


def verify(stored, predicted):
    """Match logic from competition metric."""
    stored, predicted = stored.strip(), predicted.strip()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except:
        return predicted.lower() == stored.lower()

print("Functions ready.")

In [ ]:
# === Run evaluation ===

results = []
for i, row in samples.iterrows():
    n = len(results) + 1
    total = len(samples)
    print(f"[{n}/{total}] {row.qtype} id={row.id}...", end=" ", flush=True)
    
    resp = call_gemini(row.prompt)
    text = extract_response_text(resp)
    thinking = extract_thinking_text(resp)
    
    if text is None:
        print(f"ERROR: {str(resp)[:100]}")
        results.append({
            'id': row.id, 'qtype': row.qtype, 'answer': row.answer,
            'predicted': 'ERROR', 'correct': False,
            'response': str(resp)[:500], 'thinking': None
        })
        continue
    
    predicted = extract_answer(text)
    correct = verify(str(row.answer), predicted)
    symbol = '\u2713' if correct else '\u2717'
    print(f"expected={row.answer} | predicted={predicted} | {symbol}")
    
    results.append({
        'id': row.id, 'qtype': row.qtype, 'answer': str(row.answer),
        'predicted': predicted, 'correct': correct,
        'response': text, 'thinking': thinking
    })

rdf = pd.DataFrame(results)
print(f"\nDone! {rdf.correct.sum()}/{len(rdf)} correct ({rdf.correct.mean():.0%})")

## Results Summary

In [ ]:
# === Score by type ===
print(f"Overall: {rdf.correct.sum()}/{len(rdf)} = {rdf.correct.mean():.0%}\n")

summary = rdf.groupby('qtype').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum'),
    accuracy=('correct', 'mean')
).sort_values('accuracy', ascending=False)
summary['accuracy'] = summary['accuracy'].map('{:.0%}'.format)
summary

In [ ]:
# === Wrong answers detail ===
wrong = rdf[~rdf.correct][['id', 'qtype', 'answer', 'predicted']]
if len(wrong):
    print(f"Wrong answers ({len(wrong)}):\n")
    for _, r in wrong.iterrows():
        print(f"  [{r.qtype}] id={r.id}  expected='{r.answer}'  got='{r.predicted}'")
else:
    print("All correct!")

## Inspect Individual Responses

Look at the full model response for any row.

In [ ]:
# Change idx to inspect different rows
# Or filter: rdf[rdf.qtype == 'bit_manipulation']
# Or wrong only: rdf[~rdf.correct]

idx = 0  # <-- change this
row = rdf.iloc[idx]

print(f"=== Type: {row.qtype} | ID: {row.id} ===")
print(f"Expected: {row.answer}")
print(f"Predicted: {row.predicted}")
print(f"Correct: {row.correct}")
print(f"\n{'='*60}")
print(f"MODEL RESPONSE:\n")
print(row.response)

In [ ]:
# === Inspect thinking trace (if available) ===

idx = 0  # <-- change this
row = rdf.iloc[idx]

if row.thinking:
    print(f"=== Thinking for {row.qtype} id={row.id} ===")
    print(row.thinking[:5000])
    if len(str(row.thinking)) > 5000:
        print(f"\n... [{len(row.thinking) - 5000} more chars]")
else:
    print("No thinking trace available.")

In [ ]:
# === Save results to JSONL for later use ===
output_path = WORK_DIR / 'gemini_eval_results.jsonl'
rdf.to_json(output_path, orient='records', lines=True)
print(f"Saved {len(rdf)} results to {output_path}")

In [ ]:
# === Save all results ===
output_path = WORK_DIR / 'multi_model_eval_results.jsonl'
all_rdf.to_json(output_path, orient='records', lines=True)
print(f"Saved {len(all_rdf)} results to {output_path}")

# === Wrong answers by model ===
for model_name in MODELS_TO_TEST.keys():
    mdf = all_rdf[(all_rdf.model == model_name) & (~all_rdf.correct)]
    if len(mdf):
        print(f"\n{model_name} wrong ({len(mdf)}):")
        for _, r in mdf.iterrows():
            print(f"  [{r.qtype}] expected='{r.answer}' got='{r.predicted}'")

In [ ]:
# === Comparison table: model × type ===
pivot = all_rdf.pivot_table(
    index='model', columns='qtype', values='correct', aggfunc=['sum', 'count']
)
# Build clean summary
summary_rows = []
for model_name in MODELS_TO_TEST.keys():
    mdf = all_rdf[all_rdf.model == model_name]
    row = {'model': model_name}
    for qtype in sorted(mdf.qtype.unique()):
        sub = mdf[mdf.qtype == qtype]
        row[qtype] = f"{sub.correct.sum()}/{len(sub)}"
    row['overall'] = f"{mdf.correct.sum()}/{len(mdf)} ({mdf.correct.mean():.0%})"
    summary_rows.append(row)

comparison = pd.DataFrame(summary_rows).set_index('model')
comparison

In [ ]:
# === Run all models ===
all_results = []
for model_name, model_id in MODELS_TO_TEST.items():
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({model_id})")
    print(f"{'='*60}")
    model_results = run_eval(model_name, model_id, samples)
    all_results.extend(model_results)
    
    # Quick summary
    correct = sum(r['correct'] for r in model_results)
    print(f"\n  {model_name}: {correct}/{len(model_results)} = {correct/len(model_results):.0%}")

all_rdf = pd.DataFrame(all_results)
print(f"\nTotal results: {len(all_rdf)}")

In [ ]:
def call_model(prompt, model_id):
    """Call any model on the Gemini API."""
    gen_config = {
        "temperature": 1.0,
        "maxOutputTokens": MAX_OUTPUT_TOKENS,
    }
    # Only add thinking for Gemini Pro (Gemma doesn't support it)
    if "gemini" in model_id:
        gen_config["thinkingConfig"] = {"thinkingBudget": THINKING_BUDGET}
    
    body = json.dumps({
        "contents": [{"parts": [{"text": prompt + SUFFIX}]}],
        "generationConfig": gen_config
    })
    result = subprocess.run([
        "curl", "-s", "--max-time", "180",
        "--resolve", f"generativelanguage.googleapis.com:443:{RESOLVE_IP}",
        f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent?key={API_KEY}",
        "-H", "Content-Type: application/json",
        "-d", body
    ], capture_output=True, text=True)
    try:
        return json.loads(result.stdout)
    except:
        return {"error": result.stdout[:500]}

def run_eval(model_name, model_id, samples_df):
    """Run eval on all samples for a single model."""
    results = []
    for i, row in samples_df.iterrows():
        n = len(results) + 1
        total = len(samples_df)
        print(f"  [{n}/{total}] {row.qtype}...", end=" ", flush=True)
        
        resp = call_model(row.prompt, model_id)
        text = extract_response_text(resp)
        
        if text is None:
            print(f"ERROR")
            results.append({
                'model': model_name, 'id': row.id, 'qtype': row.qtype,
                'answer': str(row.answer), 'predicted': 'ERROR', 'correct': False
            })
            continue
        
        predicted = extract_answer(text)
        correct = verify(str(row.answer), predicted)
        symbol = '\u2713' if correct else '\u2717'
        print(f"{row.answer} → {predicted} {symbol}")
        
        results.append({
            'model': model_name, 'id': row.id, 'qtype': row.qtype,
            'answer': str(row.answer), 'predicted': predicted, 'correct': correct
        })
    return results

print("Multi-model eval functions ready.")

---
## Multi-Model Comparison (Gemini 3.1 Pro + Gemma 4)

Runs the same samples through all models and compares.